In [1]:
import cobra

import pandas as pd

from Bio.Seq import Seq
from Bio import SeqIO
from Bio.Alphabet import generic_dna

import multiprocessing
from multiprocessing import Pool
from tqdm import tqdm

import scipy.stats as st
import urllib
# import seaborn as sns
# import matplotlib.pyplot as plt
import numpy as np
import warnings
import gc
from tqdm import tqdm

import requests, sys, json, re
sys.path.insert(1, '../../scripts/') # comment out in python script
from utils.load_environmental_variables import *
prebuild = '/data2/hratch/human_me/prebuild/'
# human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_model.json')

# MANE Select

In [2]:
# MANE SELECTED transcripts and protein sequences
mane_ids = pd.read_csv(prebuild + 'sequence_information/MANE.GRCh38.v0.9.summary.txt', sep = '\t')

psim_me = mane_ids.loc[:, ['Ensembl_Gene', 'HGNC_ID', 'Ensembl_nuc', 'Ensembl_prot', 'symbol', 'RefSeq_nuc', 'RefSeq_prot', '#NCBI_GeneID']]
psim_me.columns = ['ENSG_ID', 'HGNC_ID', 'ENST_ID', 'ENSP_ID', 'GENE_SYMBOL', 'REFT_ID', 'REFP_ID', 'GeneID']
psim_me = psim_me[['HGNC_ID', 'ENSG_ID', 'ENST_ID', 'ENSP_ID', 'REFT_ID', 'REFP_ID', 'GENE_SYMBOL', 'GeneID']]
psim_me['Source'] = 'MANE Select'


psim_me.drop(index = psim_me[psim_me.HGNC_ID.isna()].index, inplace = True)

sequence mapping

In [3]:
# add protein sequences
protein = list(SeqIO.parse(prebuild + 'sequence_information/MANE.GRCh38.v0.9.select_ensembl_protein.faa', "fasta"))
p_map = dict()
for p in protein:
    p_map[p.id] = str(p.seq)
psim_me['PROTEIN_SEQ'] = psim_me.ENSP_ID.map(p_map)

# add mrna sequence
mrna = list(SeqIO.parse(prebuild + 'sequence_information/MANE.GRCh38.v0.9.select_ensembl_rna.fna', "fasta"))
m_map = dict()
for m in mrna:
    m_map[m.id] = str(m.seq)
psim_me['MRNA_SEQ'] = psim_me.ENST_ID.map(m_map)

del protein
del mrna
psim_me.drop(index=psim_me[psim_me[['PROTEIN_SEQ', 'MRNA_SEQ']].isna().apply(lambda x: x.any(), axis = 1)].index.tolist(), 
             inplace = True)

# GET Refseq Select
MANE alone will not cover all the machinery

In [4]:
import gffpandas.gffpandas as gffpd
refseq_gff = gffpd.read_gff3(prebuild + 'sequence_information/GCF_000001405.39_GRCh38.p13_genomic.gff')

# parse gff
attr = refseq_gff.df.attributes.apply(lambda x: [i.split('=')[0] for i in x.split(';')])
attr_set = list()
for a in attr:
    attr_set += a
attr_set = sorted(set(attr_set))

# # initialize
# for col in attr_set:
#     refseq_gff_df[col] = float('nan')

rdf = refseq_gff.df.copy()

for col in tqdm(attr_set):
    def get_attr(x, col = col):
        if col in x:
            try:
                attrs = x.split(';')
                return attrs[[i.split('=')[0] for i in attrs].index(col)].split('=')[1]
            except:
                return float('nan')
        else:
            return float('nan')

    rdf[col] =  rdf.attributes.apply(lambda x: get_attr(x))

# filter for SELECT    
rdf = rdf[rdf.tag.notna()]
del refseq_gff
rdf.to_csv('/data2/hratch/human_me/temp_refseq.gz', compression='gzip')

100%|██████████| 93/93 [04:50<00:00,  3.12s/it]


In [5]:
rs = rdf[(rdf.tag == 'RefSeq Select') & (rdf.type != 'exon')] # just refseq select
rs.drop_duplicates(subset = ['seq_id', 'type', 'protein_id', 'Dbxref'], inplace = True) # lots of the same protein

# get rid of those already ins psim me
drop_idx = rs[rs.transcript_id.isin(psim_me.REFT_ID)].index.tolist() + rs[rs.protein_id.isin(psim_me.REFP_ID)].index.tolist()
rs.drop(index = drop_idx, inplace = True)

rs = rs[['seq_id', 'transcript_id', 'protein_id', 'type', 'Dbxref']]

# parse ids dbxref
attr = rs.Dbxref.apply(lambda x: [i.split(':')[0] for i in x.split(',')])
attr_set = list()
for a in attr:
    attr_set += a
attr_set = sorted(set(attr_set))

for col in tqdm(attr_set):
    def get_attr(x, col = col):
        if col in x:
            try:
                attrs = x.split(',')
                return ':'.join(attrs[[i.split(':')[0] for i in attrs].index(col)].split(':')[1:])
            except:
                return float('nan')
        else:
            return float('nan')

    rs[col] =  rs.Dbxref.apply(lambda x: get_attr(x))
rs.drop(columns = ['CCDS', 'Dbxref', 'Genbank', 'MIM'], inplace = True)

#backup = rs.copy()
rs.drop_duplicates(subset = ['protein_id', 'HGNC'], inplace = True)
rs.drop_duplicates(subset = ['transcript_id', 'HGNC'], inplace = True)

# any remaining overlap with psim_me
rs = rs[rs.HGNC.notna()]
rs.drop(index = rs[rs.HGNC.isin(psim_me.HGNC_ID)].index.tolist(), 
        inplace = True)

# format
rs.reset_index(inplace = True, drop = True)

unique_ids = rs.HGNC.unique().tolist()

rs.drop(columns = ['seq_id', 'type', 'GeneID'], inplace = True)

rs_ = pd.DataFrame(columns = rs.columns)
counter = 0
exceptions = list()
for hgnc in tqdm(unique_ids):
    temp = rs[rs.HGNC == hgnc]
    rs_.loc[counter, :] = [temp.transcript_id.dropna().values.tolist()[0],  temp.protein_id.dropna().values.tolist()[0], 
                           hgnc]
    counter += 1
rs = rs_
del rs_

rs['Source'] = 'RefSeq Select'
rs.columns = ['REFT_ID', 'REFP_ID', 'HGNC_ID', 'Source']

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/ipykernel_launcher.py:2 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/pandas/core/frame.py:3997 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
100%|██████████| 2952/2952 [00:07<00:00, 385.17it/s]


# REFSEQ MRNA and PROTEIN Sequence

In [6]:
# sequenc mapping
mane_mrna = list(SeqIO.parse(prebuild + 'sequence_information/MANE.GRCh38.v0.9.select_ensembl_rna.fna', "fasta"))
mane_mrna_map = {m.id: str(m.seq.transcribe()) for m in mane_mrna}
psim_me['MRNA_SEQ'] = psim_me.ENST_ID.map(mane_mrna_map)

mane_protein = list(SeqIO.parse(prebuild + 'sequence_information/MANE.GRCh38.v0.9.select_ensembl_protein.faa', "fasta"))
mane_protein_map = {p.id: str(p.seq) for p in mane_protein} # have to parse
psim_me['PROTEIN_SEQ'] = psim_me.ENSP_ID.map(mane_protein_map)

#######
rs_mrna = list(SeqIO.parse(prebuild + 'sequence_information/GCF_000001405.39_GRCh38.p13_rna.fna', "fasta"))
rs_mrna_map = {m.id: str(m.seq.transcribe()) for m in rs_mrna}
rs['MRNA_SEQ'] = rs['REFT_ID'].map(rs_mrna_map)

rs_protein = list(SeqIO.parse(prebuild + 'sequence_information/GCF_000001405.39_GRCh38.p13_protein.faa', "fasta"))
rs_protein_map = {p.id.split('.')[0]: str(p.seq) for p in rs_protein} # have to parse
rs['PROTEIN_SEQ'] = rs['REFP_ID'].apply(lambda x: x.split('.')[0]).map(rs_protein_map)

#######
psim_me.drop(index=psim_me[psim_me[['PROTEIN_SEQ', 'MRNA_SEQ']].isna().apply(lambda x: x.any(), axis = 1)].index.tolist(), 
             inplace = True)
rs.drop(index=rs[rs[['PROTEIN_SEQ', 'MRNA_SEQ']].isna().apply(lambda x: x.any(), axis = 1)].index.tolist(), 
             inplace = True)

# Merge MANE and REFSEQ PSIMs

In [7]:
# get ENSG IDs for REFSEQ
import mygene
id_map = pd.read_csv(prebuild + 'sequence_information/identifiers.txt', sep = '\t')

mg = mygene.MyGeneInfo()
mapper = mg.querymany(qterms = rs.REFT_ID.tolist(), scopes='refseq.rna', fields='ensembl.gene', species='human', 
            as_dataframe = True)

mapper = mapper[['ensembl.gene']]
mapper['refseq.transcript'] = mapper.index
mapper.reset_index(inplace = True, drop = True)
mapper.columns = ['ENSG_ID', 'REFT_ID']
mapper.to_csv(prebuild + 'sequence_information/map_ensg_refseqrna.csv')

rs['ENSG_ID'] = rs.REFT_ID.map(dict(zip(mapper.REFT_ID, mapper.ENSG_ID)))

mapper = id_map[['Ensembl gene ID', 'HGNC ID']]
mapper = mapper[mapper['HGNC ID'].isin(rs.HGNC_ID)]

tmp = rs[rs.ENSG_ID.isna()].copy()
tmp['ENSG_ID'] = rs[rs.ENSG_ID.isna()].HGNC_ID.map(dict(zip(mapper['HGNC ID'], mapper['Ensembl gene ID'])))
tmp.drop(index = tmp[(tmp.ENSG_ID.isin(rs.ENSG_ID)) | (tmp.ENSG_ID.isin(psim_me.ENSG_ID))].index, inplace = True)
rs.loc[tmp.index,'ENSG_ID'] = tmp.ENSG_ID.tolist()

rs.drop(index = rs[rs.ENSG_ID.isna()].index, inplace = True)

# get gene symbol
mapper = mg.querymany(qterms = rs.REFT_ID.tolist(), scopes='refseq.rna', fields='symbol', species='human', 
            as_dataframe = True)
mapper = pd.DataFrame(mapper[mapper['symbol'].notna()]['symbol'])
mapper.to_csv(prebuild + 'sequence_information/map_symbol_refseqrna.csv')

rs['GENE_SYMBOL'] = rs.REFT_ID.map(dict(zip(mapper.index, mapper.symbol)))



querying 1-1000...done.
querying 1001-2000...done.
querying 2001-2952...done.
Finished.
querying 1-1000...

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/biothings_client/base.py:143 FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead


done.
querying 1001-2000...done.
querying 2001-2937...done.
Finished.


In [8]:
for col in set(psim_me.columns).difference(rs.columns):
    rs[col] = float('nan')
rs = rs[psim_me.columns.tolist()]
psim_me = pd.concat([psim_me, rs], axis = 0)


# forgot to check for '.' in ensg
test = psim_me[psim_me.Source == 'MANE Select']
test.ENSG_ID = test.ENSG_ID.apply(lambda x: x.split('.')[0])
test = test[test.ENSG_ID.isin(rs.ENSG_ID)]
psim_me.drop(index = psim_me[psim_me.ENSG_ID.isin(test.ENSG_ID.tolist())].index, inplace = True)

psim_me.reset_index(inplace = True, drop = True)
psim_me.to_csv('/data2/hratch/human_me/temp_psim.csv')

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/pandas/core/generic.py:5303 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


# Load metabolic and expression machinery

In [378]:
# from "preanalyses/expression_machinery_list.ipynb"
expression_machinery = sorted(open(build_files_path + 'expression_machinery.txt').read().splitlines())

m_model = cobra.io.read_sbml_model(input_data_path + 'recon2_2.xml')
metabolic_machinery = sorted([g.id if g.id.count(':') == 1 else ':'.join(g.id.split(':')[1:]) for g in m_model.genes]) 
all_mach = sorted(set(expression_machinery + metabolic_machinery))
missing_genes = set(all_mach).difference(psim_me.HGNC_ID)

# not protein-coding
rm = ['HGNC:4686', 'HGNC:31007', 'HGNC:10031', 'HGNC:33511']
for gene_list in [expression_machinery, metabolic_machinery, all_mach, missing_genes]:
    for hgnc_id in rm:
        if hgnc_id in gene_list:
            gene_list.remove(hgnc_id)

In [304]:
# map missing genes to gene symbol
id_map = pd.read_csv(prebuild + 'sequence_information/identifiers.txt', sep = '\t')
mapper = id_map[id_map["HGNC ID"].isin(missing_genes)]

if mapper['HGNC ID'].unique().shape[0] == mapper['Approved symbol'].unique().shape[0] == mapper.shape[0]:
    if psim_me[psim_me.GENE_SYMBOL.isin(mapper['Approved symbol'])].shape[0] == 0:
        mapper = dict(zip(mapper['HGNC ID'], mapper['Approved symbol']))
else:
    raise ValueError('Non-unique ids when mapping missing genes')
mapper = {mapper[g]:g  for g in missing_genes}

# Add missing machinery via Appris

In [305]:
# format appris data
appris_seq = list(SeqIO.parse(prebuild + 'sequence_information/appris_data.transl.fa', "fasta"))
appris_pid = {entry.id.split('|')[1]: entry.id.split('|')[0] for entry in appris_seq}
appris_prot = {entry.id.split('|')[1]: str(entry.seq) for entry in appris_seq}

appris = pd.read_csv(prebuild + 'sequence_information/appris_data.principal.txt', sep = '\t', header = None)
appris.columns = ['GENE_SYMBOL', 'ENSG_ID', 'ENST_ID', 'OTHER', 'APPRIS_CAT']
appris.drop(columns = ['OTHER'], inplace = True)
appris = appris[appris.GENE_SYMBOL.isin(list(mapper.keys()))]
appris[['APPRIS_CAT', 'APPRIS_SCORE']] = appris.APPRIS_CAT.str.split(':', expand = True)
appris.APPRIS_SCORE = appris.APPRIS_SCORE.astype(float)
appris['ENSP_ID'] = appris.ENST_ID.map(appris_pid)
appris['PROTEIN_SEQ'] = appris.ENST_ID.map(appris_prot)
appris = appris[appris.APPRIS_CAT == 'PRINCIPAL']

In [306]:
res = pd.DataFrame(columns = appris.columns)
for gs in tqdm(mapper.keys()):
    temp = appris[appris.GENE_SYMBOL == gs]
    if temp.shape[0] > 0:
        temp = pd.DataFrame(temp[(temp.APPRIS_SCORE == temp.APPRIS_SCORE.max())].iloc[0,:]).T
        res = pd.concat([res, temp], axis = 0, ignore_index = True)

# mapper = {v:k for k,v in mapper.items()}
res['HGNC_ID'] = res.GENE_SYMBOL.map(mapper)

res.drop(columns = ['APPRIS_CAT', 'APPRIS_SCORE'], inplace = True)
res['Source'] = 'APPRIS PRINCIPAL'

appris = res
del res
appris.reset_index(inplace = True, drop = True)

for col in appris.columns:
    if appris[appris[col].isin(psim_me[col])].shape[0]>0:
        raise ValueError('Something mapped non-uniquely')

100%|██████████| 82/82 [00:00<00:00, 300.22it/s]


In [307]:
# # get mrna seq for appris from the ENST ID using the rest API
def get_mrna_seq(enst_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + enst_id + '?type=cdna' 
        x = requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text # introns and UTRs
        return str(Seq(x).transcribe())
    except:
        return float('nan')
    
from pandarallel import pandarallel
pandarallel.initialize(nb_workers = 10, verbose = 0)

In [309]:
fail_mssg = 'You have exceeded uhe limiu of 15 requesus per second; please reduce your concurrenu connecuions'
appris['MRNA_SEQ'] = fail_mssg
fail_idx = appris.index

fail = True
counter = 1


while fail and (counter < 11):
    print('Iteration: {}'.format(counter))
    
    if counter > 7:
        pandarallel.initialize(nb_workers = 2, verbose = 0)
    elif counter > 4:
        pandarallel.initialize(nb_workers = 5, verbose = 0)

    appris_temp = appris.loc[fail_idx, :]
    appris_temp.ENST_ID = appris_temp.ENST_ID.apply(lambda x: x.split('.')[0])
    appris_temp['MRNA_SEQ'] = appris_temp.ENST_ID.parallel_apply(lambda x: get_mrna_seq(x))
    appris.loc[fail_idx, 'MRNA_SEQ'] = appris_temp.MRNA_SEQ.tolist()
    
    fail_idx = appris[appris['MRNA_SEQ'] == fail_mssg].index
    fail = (fail_idx.shape[0] > 0)
    print('Missing sequences: {}'.format(fail_idx.shape[0]))
    print(fail)

    counter += 1
    print('---------------')

for col in appris.columns:
    if appris[appris[col].isin(psim_me[col])].shape[0] > 0:
        raise ValueError('Something mapped non-uniquely')

Iteration: 1
Missing sequences: 2
True
---------------
Iteration: 2
Missing sequences: 0
False
---------------


In [315]:
# merge with PSIM ME
for col in set(psim_me.columns).difference(appris.columns):
    appris[col] = float('nan')
appris = appris[psim_me.columns.tolist()]
psim_me = pd.concat([psim_me, appris], axis = 0, ignore_index=True)

# Add Remainder of Missing Genes Manually

In [338]:
def get_protein_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=protein;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

# manually get remaining
missing_genes = sorted(set(missing_genes).difference(psim_me.HGNC_ID))
missing = pd.DataFrame(columns = appris.columns)
missing['HGNC_ID'] = missing_genes
missing['GENE_SYMBOL'] = missing.HGNC_ID.map({v:k for k,v in mapper.items()})
missing['Source'] = 'Manual'
missing.index = missing.GENE_SYMBOL
missing.loc['RMRP', ['ENSG_ID', 'ENST_ID']] = ['ENSG00000277027.1', 'ENST00000363046']
missing.loc['ELOA3DP', ['ENSG_ID', 'ENST_ID', 'ENSP_ID', 'PROTEIN_SEQ']] = ['ENSG00000183791.5', 
            'ENST00000330682.3', 'ENSP00000328232', 
                'MAAGSTTLRAVGKLQVRLATKTEPKKLEKYLQKLSALPMTADILAETGIRKTVKRLRKHQHVGDFARDLAARWKKLVLVDRNTGPDPQDPEESASRQRFGEALQEREKAWGFPENATAPRSPSHSPEHRRTARRTPPGQQRPHPRSPSREPRAERKRPRMAPADSGPHRDPPTRTAPLPMPEGPEPAVPGEQPGRGHAHAAQGGPLLGQGCQGQPQGEAVGSHSKGHKSSRGASAQKSPPVQESQSERLQAAVADSAGPKTVPSHVFSELWDPSEAWMQANYDLLSAFEAMTSQANPEALSAPTLQEEAAFPGRRVNAKMPVYSGSRPACQLQVPTLRQQCLRVPRNNPDALGDVEGVPYSALEPVLEGWTPDQPYRTEKDNAALARETDELWRIHCLQDFKEEKPQEHESWRELYLRLRDAREQRLRVVTTKIRSARENKPSGRQTKMICFNSVAKTPYDASRRQEKSAGAADPGNGEMEPAPKPAGSSQAPSGLGDGDGGSVSGGGSSNRHAAPADKTRKQAAKKVAPLMAKAIRDYKGRFSRR']
missing.loc['ELOA3CP', ['ENSG_ID', 'ENST_ID']] = ['ENSG00000275553', 'ENST00000620522.2']
missing.reset_index(inplace = True, drop = True)

missing.loc[0, 'MRNA_SEQ'] = get_mrna_seq('ENST00000620522')
missing.loc[0, 'PROTEIN_SEQ'] = get_protein_seq('ENST00000620522')

In [439]:
psim_me = pd.concat([psim_me, missing], axis = 0, ignore_index = True)
if len(set(all_mach).difference(psim_me.HGNC_ID)) > 0:
    raise ValueError('All machinery should be present in PSIM ME at this point')
psim_me.to_csv('/data2/hratch/human_me/temp_psim.csv')

# Make sure all values are present

In [470]:
def test_psim(psim_me):
    test = psim_me.copy()
    test.index = test.HGNC_ID
    test = test.loc[all_mach, :]
    for col in test.columns:
        if test[col].dropna().unique().shape[0] != test[col].dropna().shape[0]:
            print('Non-unique values: ' + col)

    for col in ['PROTEIN_SEQ', 'MRNA_SEQ', 'HGNC_ID', 'ENSG_ID']:
        if test[test[col].isna()].shape[0] > 0:
            print('Missing required values: ' + col)

In [471]:
test_psim(psim_me)

Non-unique values: Source
Non-unique values: PROTEIN_SEQ
Non-unique values: MRNA_SEQ


# Add PREMRNA Sequence 

In [467]:
from pandarallel import pandarallel
pandarallel.initialize(nb_workers = 20, verbose = 0, progress_bar = True)

psim_me = pd.read_csv('/data2/hratch/human_me/temp_psim.csv', index_col = 0)

def get_premrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
        x = requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text # introns and UTRs
        return str(Seq(x).transcribe())
    except:
        return float('nan')

In [473]:
fail_mssg = 'You have exceeded uhe limiu of 15 requesus per second; please reduce your concurrenu connecuions'
psim_me['PREMRNA_SEQ'] = fail_mssg
fail_idx = psim_me.index
print('Begin getting sequence for {} genes'.format(psim_me.shape[0]))


fail = True
counter = 1


while fail and (counter < 11):
    print('Iteration: {}'.format(counter))
    
    if counter > 7:
        pandarallel.initialize(nb_workers = 5, verbose = 0)
    elif counter > 4:
        pandarallel.initialize(nb_workers = 10, verbose = 0)

    psim_temp = psim_me.loc[fail_idx, :]
    psim_temp.ENSG_ID = psim_temp.ENSG_ID.apply(lambda x: x.split('.')[0])
    psim_temp['PREMRNA_SEQ'] = psim_temp.ENSG_ID.parallel_apply(lambda x: get_premrna_seq(x))
    psim_me.loc[fail_idx, 'PREMRNA_SEQ'] = psim_temp.PREMRNA_SEQ.tolist()
    
    fail_idx = psim_me[psim_me['PREMRNA_SEQ'] == fail_mssg].index
    fail = (fail_idx.shape[0] > 0)
    print('Missing sequences: {}'.format(fail_idx.shape[0]))
    print(fail)

    counter += 1
    print('---------------')
psim_me.to_csv('/data2/hratch/human_me/temp_psim2.csv')

In [475]:
psim_me = pd.read_csv('/data2/hratch/human_me/temp_psim2.csv', index_col = 0)

In [483]:
def test_psim(psim_me):
    test = psim_me.copy()
    test.index = test.HGNC_ID
    test = test.loc[all_mach, :]
    for col in test.columns:
        if test[col].dropna().unique().shape[0] != test[col].dropna().shape[0]:
            print('Non-unique values: ' + col)

    for col in ['PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ','HGNC_ID', 'ENSG_ID']:
        if test[test[col].isna()].shape[0] > 0:
            print('Missing required values: ' + col)
            
    

In [484]:
test_psim(psim_me)

Non-unique values: Source
Non-unique values: PROTEIN_SEQ
Non-unique values: MRNA_SEQ


In [489]:
premrna_l = psim_me.PREMRNA_SEQ.apply(lambda x: len(x))
mrna_l = psim_me.MRNA_SEQ.apply(lambda x: len(x))
protein_l = psim_me.PROTEIN_SEQ.apply(lambda x: len(x))

# check lengths make sense
fail = psim_me[mrna_l > premrna_l]
fail = fail[fail.HGNC_ID.isin(all_mach)]
if (fail.shape[0]>1) or (fail.HGNC_ID.values.tolist()[0] != 'HGNC:30771'):
    raise ValueError('Did this part manually, only the above gene popped up. If this error is raised, double check')

    
idx = fail.index
psim_me.loc[idx, ['ENST_ID', 'MRNA_SEQ', 'PREMRNA_SEQ']] = ['ENST00000332567', get_mrna_seq('ENST00000332567'),  
                                                            get_premrna_seq('ENST00000332567')]

fail = psim_me[mrna_l < (3*protein_l)]
if fail.shape[0] > 0:
    raise ValueError('Protein lengths do not agree with mrna lengths')
# psim_me.to_csv('/data2/hratch/human_me/temp_psim2.csv')

# Add Other Features

In [487]:
# polyA - from "preanalyses/polyA_statistics.ipynb"
polyA = pd.read_csv(build_files_path + 'polyA_length.csv', index_col = 0)
psim_me['POLYA_LENGTH'] = psim_me.GENE_SYMBOL.map(dict(zip(polyA.index, polyA.MEAN)))

In [682]:
# sec features
sec_psim = pd.read_csv(prebuild + 'recon2_2s_psim_human.tab', sep = '\t')

# map to uniprot to work with sec_psim
mapper = id_map[['HGNC ID', 'UniProt accession']]
mapper.columns = ['HGNC_ID', 'Uniprot']

mapper = mapper[(mapper.HGNC_ID.isin(psim_me.HGNC_ID)) & (mapper.Uniprot.isin(sec_psim.Entry))]
mapper = dict(zip(mapper.HGNC_ID, mapper.Uniprot))
# mapper = {k:v for k,v in mapper.items() if v in tmp and k in tmp2}

# do the mapping
sec_psim.index = sec_psim.Entry
cols = ['Entry', 'SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'Location']
pandarallel.initialize(nb_workers = 20, verbose = 0, progress_bar = True)
sec_features = psim_me.HGNC_ID.parallel_apply(lambda x: sec_psim.loc[mapper[x], cols].tolist() if x in mapper.keys() else [float('nan')]*8)
sec_features=pd.DataFrame(sec_features.tolist())
sec_features.columns = cols
sec_features.rename(columns = {'Entry': 'UNIPROT_ID'}, inplace = True)
psim_me = pd.concat([psim_me, sec_features], axis = 1)

In [746]:
# ptrs - built and used in preanalyses/process_PTR.py
ptr = pd.read_csv(build_files_path + 'PTR_Gagneur_processed.tsv', sep = '\t', index_col = 0)
psim_me['PTR'] = psim_me.HGNC_ID.map(dict(zip(ptr.index, ptr.Median)))
psim_me['PTR_TISSUE'] = 'Median'
psim_me['CONSTANT_PTR'] = False
psim_me.loc[psim_me[psim_me['PTR'].isna()].index, 'PTR'] = ptr.Median.median()

# N introns

# coupling params? 